In [1]:
from src.data_loader import load_and_merge
from src.preprocessing import Preprocessor

df = load_and_merge("./data/movies.csv", "./data/ratings.csv")

pre = Preprocessor()
df = pre.encode_ids(df)

user_item = pre.create_user_item_matrix(df)

print(user_item.shape)

(611, 32918)


Matrix Factorization Training

In [4]:
from models.matrix_factorization import MatrixFactorization
from src.preprocessing import Preprocessor
from src.data_loader import load_and_merge

df = load_and_merge("./data/movies.csv", "./data/ratings.csv")

pre = Preprocessor()
df = pre.encode_ids(df)

n_users, n_items = pre.get_num_users_items(df)

mf = MatrixFactorization(
    n_users,
    n_items,
    n_factors=40,
    epochs=20,
    lr=0.005,
    reg=0.05
)

mf.fit(df)

print("Prediction (user 0, item 0):", mf.predict(0, 0))

Epoch 1/20, WeightedLoss: 104212.95, MeanWeightedMSE: 0.8370
Epoch 2/20, WeightedLoss: 94810.71, MeanWeightedMSE: 0.7615
Epoch 3/20, WeightedLoss: 91490.45, MeanWeightedMSE: 0.7348
Epoch 4/20, WeightedLoss: 89437.38, MeanWeightedMSE: 0.7183
Epoch 5/20, WeightedLoss: 87944.02, MeanWeightedMSE: 0.7063
Epoch 6/20, WeightedLoss: 86767.62, MeanWeightedMSE: 0.6969
Epoch 7/20, WeightedLoss: 85794.85, MeanWeightedMSE: 0.6890
Epoch 8/20, WeightedLoss: 84963.22, MeanWeightedMSE: 0.6824
Epoch 9/20, WeightedLoss: 84234.62, MeanWeightedMSE: 0.6765
Epoch 10/20, WeightedLoss: 83583.95, MeanWeightedMSE: 0.6713
Epoch 11/20, WeightedLoss: 82993.74, MeanWeightedMSE: 0.6666
Epoch 12/20, WeightedLoss: 82451.32, MeanWeightedMSE: 0.6622
Epoch 13/20, WeightedLoss: 81947.11, MeanWeightedMSE: 0.6581
Epoch 14/20, WeightedLoss: 81473.66, MeanWeightedMSE: 0.6543
Epoch 15/20, WeightedLoss: 81024.91, MeanWeightedMSE: 0.6507
Epoch 16/20, WeightedLoss: 80595.78, MeanWeightedMSE: 0.6473
Epoch 17/20, WeightedLoss: 80181

In [5]:
import pickle

with open("./outputs/saved_models/mf.pkl", "wb") as f:
    pickle.dump(mf, f)

Hybrid Predictor (MF + Genre Similarity)

In [6]:
import pandas as pd
from models.hybrid import HybridRecommender

movies = pd.read_csv("./data/movies.csv")

hybrid = HybridRecommender(mf, movies, alpha=0.7)

user = 0

user_history = df[df['user'] == user]['movieId'].values[:10]

item = 0  

item_movieId = pre.movie_encoder.inverse_transform([item])[0]

print(hybrid.predict(user, item_movieId, user_history))

3.36565182861758


Evaluation


In [7]:
from evaluation.evaluate import evaluate_models
from models.matrix_factorization import MatrixFactorization
from src.preprocessing import Preprocessor
from src.data_loader import load_and_merge

df = load_and_merge("./data/movies.csv", "./data/ratings.csv")

df_small = df.sample(20000, random_state=42)

pre_small = Preprocessor()
df_small = pre_small.encode_ids(df_small)

results = evaluate_models(df_small, pre_small, movies)

print(results)

Epoch 1/5, WeightedLoss: 14414.89, MeanWeightedMSE: 0.9009
Epoch 2/5, WeightedLoss: 12978.73, MeanWeightedMSE: 0.8112
Epoch 3/5, WeightedLoss: 12233.60, MeanWeightedMSE: 0.7646
Epoch 4/5, WeightedLoss: 11721.80, MeanWeightedMSE: 0.7326
Epoch 5/5, WeightedLoss: 11328.92, MeanWeightedMSE: 0.7081
Starting SVD++...


Epoch 1, Loss: 14422.18


Epoch 2, Loss: 12945.98
Finished SVD++
                 model      RMSE       MAE
0             Baseline  0.981266  0.751802
1  MatrixFactorization  0.896780  0.702925
2                SVD++  0.918344  0.727018
3                  KNN  0.971576  0.747896
4               Hybrid  0.921794  0.740581
